## Current scope: real PDFs and a real LLM

The project has returned to its original extension: visibility-diff plus a transformer classifier protecting an actual LLM screener on real PDF résumés. Start with [Real PDF LLM Defence — Colab](colab/Real_PDF_LLM_Defence_Colab.ipynb), and optionally [T4 classifier training](colab/Train_Transformer_Classifier_T4.ipynb).

The earlier text-classification results and paper are a **pilot**. They are not results for the real-PDF/LLM extension. The new experiment must be run in Colab before its M4–M6 results and final paper can be completed. This notice supersedes earlier statements that narrowed the project to text-only detection.

# M3 — Research gap, hypothesis and experiment plan

**Scope:** small, CPU-only NLP detector; no LLM training or paid API.

The base paper is Jay Barach, *ResumeShield*, [arXiv:2609.20188v1](https://arxiv.org/html/2609.20188v1). See the review notes added to M1 and M2 before using their historical claims. M2 is a partial reconstruction and M3 is a narrower extension than the original PDF/LLM proposal.

## Research gap

The anchor's small synthetic evaluation does not establish whether a trained text detector generalizes to held-out directive families while avoiding benign instruction-like resume content. This is a bounded evaluation gap, not a claim that trained injection detection is new.

The extension targets visible paraphrases and benign security-related sentences. It does not claim to solve hidden-text extraction or establish a new classifier architecture. PhantomLint already studies document visibility, so visibility comparison itself is not our novelty.

## One hypothesis and prediction recorded before training

On the frozen held-out synthetic test set, a normalized character TF-IDF logistic-regression detector will improve injection recall by at least 0.20 absolute over a fixed keyword baseline, with a clean-text false-positive rate no greater than 0.10.

**Prediction:** recall gain ≥20 percentage points and FPR ≤10%; both conditions must hold. The numeric prediction was saved in [M3_Protocol.json](M3_Protocol.json) before the first training call. This notebook is a later readable rendering of that record. The run records a hash and UTC start time; this is not an independently registered or externally timestamped protocol.

## Experimental plan

1. Freeze 80 authored English sentences: 32 train, 16 validation, 32 test, balanced by class in each split.
2. Keep directive/benign families separate across splits. Make original, zero-width, and homoglyph versions inside each split: 96/48/96 rows.
3. Fit character TF-IDF and logistic regression on training rows only.
4. Select the threshold using validation only, under the fixed FPR rule.
5. Evaluate the fixed keyword baseline and learned detector on the same test rows.
6. Remove normalization, replace character features with word features, and test normalized keywords.
7. Report confusion counts, precision/recall/F1/FPR, family slices, original-only metrics and all failures.
8. Retain the result even if the hypothesis fails; do not change the test set or threshold afterward.

**Controls:** same labels, splits, train data, classifier settings, and threshold-selection procedure. Word-feature ablation reselects its threshold on validation, so it compares complete pipelines, not only representations at an identical operating point. No sample-size inflation: 96 test rows correspond to 32 source sentences and eight families.

In [1]:
import json
from pathlib import Path
protocol = json.loads(Path('M3_Protocol.json').read_text())
print(json.dumps(protocol, indent=2))

{
  "study": "Lightweight detection of visible instruction injections in synthetic resume text",
  "anchor": "Jay Barach, ResumeShield, arXiv:2609.20188v1",
  "gap": "The anchor's small synthetic evaluation does not establish whether a trained text detector generalizes to held-out directive families while avoiding benign instruction-like resume content. This is a bounded evaluation gap, not a claim that trained injection detection is new.",
  "hypothesis": "On the frozen held-out synthetic test set, a normalized character TF-IDF logistic-regression detector will improve injection recall by at least 0.20 absolute over a fixed keyword baseline, with a clean-text false-positive rate no greater than 0.10.",
  "prediction": {
    "recall_gain_min": 0.2,
    "experimental_fpr_max": 0.1
  },
  "primary_comparison": [
    "keywords",
    "normalized_char_lr"
  ],
  "unit": "One independently authored sentence; encoding variants of a sentence stay in the same split and are not independent sampl